In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/sampleSubmission.csv
/kaggle/input/competitions/word2vec-nlp-tutorial/unlabeledTrainData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip


In [5]:
import os
import sys
import logging
import time
import re

import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from transformers import BertTokenizerFast, BertForSequenceClassification
from tqdm import tqdm

# ===================== 路径配置 =====================
TRAIN_ZIP_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip"
TEST_ZIP_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
SAMPLE_SUB_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/sampleSubmission.csv"
OUTPUT_CSV = "/kaggle/working/bert_submission.csv"
os.makedirs("/kaggle/working", exist_ok=True)

# ==========日志修复 ==========
root_logger = logging.getLogger()
if root_logger.handlers:
    root_logger.handlers.clear()

logging.basicConfig(
    stream=sys.stdout,
    format='%(asctime)s | %(levelname)s | %(message)s',
    level=logging.INFO
)
program = os.path.basename(sys.argv[0])
logger = logging.getLogger(program)
logger.setLevel(logging.INFO)
logger.info("=== Logger initialized successfully ===")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Running on device: {device}")
print(f"Running on device: {device}")

# ===================== 修改Dataset：接收已经encode好的数据，训练阶段不再分词 =====================
class ImdbDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# ===================== 加载数据 =====================
logger.info("Loading training data ...")
train_df = pd.read_csv(TRAIN_ZIP_PATH, header=0, delimiter="\t", quoting=3)
logger.info(f"train shape = {train_df.shape}")
print(f"train shape = {train_df.shape}")

logger.info("Loading test data ...")
test_df = pd.read_csv(TEST_ZIP_PATH, header=0, delimiter="\t", quoting=3)
logger.info(f"test shape = {test_df.shape}")
print(f"test shape = {test_df.shape}")

X_train, X_val, y_train, y_val = train_test_split(
    train_df["review"].tolist(),
    train_df["sentiment"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=train_df["sentiment"]
)

model_name = "bert-base-uncased"
tokenizer = BertTokenizerFast.from_pretrained(model_name)

max_seq_len = 128
# =========【关键优化】提前全部分词，只做一次，不在循环里分词 =========
logger.info("Pre‑tokenizing train/val texts (one‑time cost) ...")
print("Pre‑tokenizing train/val texts (one‑time cost) ...")

train_enc = tokenizer(X_train, max_length=max_seq_len, truncation=True, padding="max_length")
val_enc = tokenizer(X_val, max_length=max_seq_len, truncation=True, padding="max_length")
test_enc = tokenizer(test_df["review"].tolist(), max_length=max_seq_len, truncation=True, padding="max_length")

train_dataset = ImdbDataset(train_enc, y_train)
val_dataset = ImdbDataset(val_enc, y_val)
test_dataset = ImdbDataset(test_enc, labels=None)

# =========【关键优化】num_workers，pin_memory加速GPU拷贝 =========
batch_size = 32
num_workers = 2

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)
optimizer = optim.AdamW(model.parameters(), lr=2e-5)
epochs = 3

# ===================== 训练验证函数 =====================
def train_one_epoch(model, loader, opt, dev):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    steps = 0
    for batch in tqdm(loader):
        input_ids = batch["input_ids"].to(dev)
        attn_mask = batch["attention_mask"].to(dev)
        labels = batch["labels"].to(dev)
        opt.zero_grad()
        out = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
        loss = out.loss
        logits = out.logits
        loss.backward()
        opt.step()

        total_loss += loss.item()
        pred = torch.argmax(logits, dim=1)
        acc = accuracy_score(labels.cpu().numpy(), pred.cpu().numpy())
        total_acc += acc
        steps += 1
    return total_loss / steps, total_acc / steps

def val_one_epoch(model, loader, dev):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    steps = 0
    with torch.no_grad():
        for batch in tqdm(loader):
            input_ids = batch["input_ids"].to(dev)
            attn_mask = batch["attention_mask"].to(dev)
            labels = batch["labels"].to(dev)
            out = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
            loss = out.loss
            logits = out.logits
            total_loss += loss.item()
            pred = torch.argmax(logits, dim=1)
            acc = accuracy_score(labels.cpu().numpy(), pred.cpu().numpy())
            total_acc += acc
            steps += 1
    return total_loss / steps, total_acc / steps

# ===================== 主训练循环 =====================
logger.info("==== Start training ====")
print("==== Start training ====")

for ep in range(1, epochs+1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, device)
    va_loss, va_acc = val_one_epoch(model, val_loader, device)
    cost = time.time() - t0
    msg = f"Epoch {ep:2d} | Train Loss:{tr_loss:.4f} Acc:{tr_acc:.4f} | Val Loss:{va_loss:.4f} Acc:{va_acc:.4f} | Time:{cost:.1f}s"
    logger.info(msg)
    print(msg)

# ===================== 预测 =====================
logger.info("Start predicting test set ...")
print("Start predicting test set ...")
model.eval()
preds_all = []
with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(device)
        attn_mask = batch["attention_mask"].to(device)
        out = model(input_ids=input_ids, attention_mask=attn_mask)
        logits = out.logits
        pred = torch.argmax(logits, dim=1).cpu().numpy().tolist()
        preds_all.extend(pred)

sample = pd.read_csv(SAMPLE_SUB_PATH)
sample["sentiment"] = preds_all
sample.to_csv(OUTPUT_CSV, index=False, quoting=3)

logger.info(f"Prediction finished, saved to {OUTPUT_CSV}, shape {sample.shape}")
print(f"Prediction finished, saved to {OUTPUT_CSV}, shape {sample.shape}")


2026-08-21 01:52:05,732 | INFO | === Logger initialized successfully ===
2026-08-21 01:52:05,733 | INFO | Running on device: cuda
Running on device: cuda
2026-08-21 01:52:05,734 | INFO | Loading training data ...
2026-08-21 01:52:06,488 | INFO | train shape = (25000, 3)
train shape = (25000, 3)
2026-08-21 01:52:06,489 | INFO | Loading test data ...
2026-08-21 01:52:07,125 | INFO | test shape = (25000, 2)
test shape = (25000, 2)
2026-08-21 01:52:07,543 | INFO | HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"


2026-08-21 01:52:07,544 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-08-21 01:52:07,654 | INFO | HTTP Request: GET https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

2026-08-21 01:52:07,732 | INFO | HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-08-21 01:52:07,805 | INFO | HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-08-21 01:52:07,869 | INFO | HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-08-21 01:52:07,933 | INFO | HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-08-21 01:52:07,999 | INFO | HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/vocab.txt "HTTP/1.1 200 OK"
2026-08-21 01:52:08,067 | INFO | HTTP Request: GET https://huggingface.co/bert-base-uncased/resolve/main/vocab.tx

vocab.txt: 0.00B [00:00, ?B/s]

...

100%|██████████| 157/157 [00:37<00:00,  4.13it/s]

2026-08-21 02:17:31,826 | INFO | Epoch  3 | Train Loss:0.1152 Acc:0.9566 | Val Loss:0.3710 Acc:0.8806 | Time:506.8s
Epoch  3 | Train Loss:0.1152 Acc:0.9566 | Val Loss:0.3710 Acc:0.8806 | Time:506.8s
2026-08-21 02:17:31,827 | INFO | Start predicting test set ...


Start predicting test set ...


100%|██████████| 782/782 [03:09<00:00,  4.14it/s]


2026-08-21 02:20:40,970 | INFO | Prediction finished, saved to /kaggle/working/bert_submission.csv, shape (25000, 2)
Prediction finished, saved to /kaggle/working/bert_submission.csv, shape (25000, 2)


In [4]:
import torch
print("torch cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("device name:", torch.cuda.get_device_name(0))

torch cuda available: True
device count: 2
device name: Tesla T4


In [6]:
import os
import sys
import logging
import time
import random
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from transformers import BertTokenizerFast, BertForSequenceClassification, get_linear_schedule_with_warmup
from tqdm import tqdm

# ===================== 全局随机种子，保证可复现 =====================
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ===================== 路径配置 =====================
TRAIN_ZIP_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip"
TEST_ZIP_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
SAMPLE_SUB_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/sampleSubmission.csv"
OUTPUT_CSV = "/kaggle/working/bert_submission.csv"
BEST_MODEL_PATH = "/kaggle/working/best_bert.pt"
os.makedirs("/kaggle/working", exist_ok=True)

# ==========日志修复 ==========
root_logger = logging.getLogger()
if root_logger.handlers:
    root_logger.handlers.clear()
logging.basicConfig(
    stream=sys.stdout,
    format='%(asctime)s | %(levelname)s | %(message)s',
    level=logging.INFO
)
program = os.path.basename(sys.argv[0])
logger = logging.getLogger(program)
logger.setLevel(logging.INFO)
logger.info("=== Logger initialized successfully ===")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Running on device: {device}")
print(f"Running on device: {device}")

# ===================== Dataset：接收已经encode好的数据 =====================
class ImdbDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# ===================== 加载数据 =====================
logger.info("Loading training data ...")
train_df = pd.read_csv(TRAIN_ZIP_PATH, header=0, delimiter="\t", quoting=3)
logger.info(f"train shape = {train_df.shape}")

logger.info("Loading test data ...")
test_df = pd.read_csv(TEST_ZIP_PATH, header=0, delimiter="\t", quoting=3)
logger.info(f"test shape = {test_df.shape}")

X_train, X_val, y_train, y_val = train_test_split(
    train_df["review"].tolist(),
    train_df["sentiment"].tolist(),
    test_size=0.2,
    random_state=SEED,
    stratify=train_df["sentiment"]
)

model_name = "bert-base-uncased"
tokenizer = BertTokenizerFast.from_pretrained(model_name)
max_seq_len = 128

# 一次性全部预分词
logger.info("Pre‑tokenizing train/val/test texts ...")
train_enc = tokenizer(X_train, max_length=max_seq_len, truncation=True, padding="max_length")
val_enc = tokenizer(X_val, max_length=max_seq_len, truncation=True, padding="max_length")
test_enc = tokenizer(test_df["review"].tolist(), max_length=max_seq_len, truncation=True, padding="max_length")

train_dataset = ImdbDataset(train_enc, y_train)
val_dataset = ImdbDataset(val_enc, y_val)
test_dataset = ImdbDataset(test_enc, labels=None)

# ========= Kaggle 坑：num_workers=0！多进程容易卡死/报错 =========
batch_size = 16
num_workers = 0
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

epochs = 3
lr = 2e-5
optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

# warmup调度器
total_steps = len(train_loader) * epochs
warmup_steps = int(total_steps * 0.1)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

scaler = GradScaler()  # 混合精度AMP
grad_clip_norm = 1.0   # 梯度裁剪

# ===================== 训练验证函数（加入AMP混合精度） =====================
def train_one_epoch(model, loader, opt, sch, scaler, dev):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    steps = 0
    pbar = tqdm(loader, desc="Train")
    for batch in pbar:
        input_ids = batch["input_ids"].to(dev)
        attn_mask = batch["attention_mask"].to(dev)
        labels = batch["labels"].to(dev)
        opt.zero_grad()

        with autocast():
            out = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
            loss = out.loss
            logits = out.logits

        scaler.scale(loss).backward()
        # 梯度裁剪
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)

        scaler.step(opt)
        scaler.update()
        sch.step()

        total_loss += loss.item()
        pred = torch.argmax(logits, dim=1)
        acc = accuracy_score(labels.cpu().numpy(), pred.cpu().numpy())
        total_acc += acc
        steps += 1
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
    return total_loss / steps, total_acc / steps

def val_one_epoch(model, loader, dev):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    steps = 0
    with torch.no_grad():
        pbar = tqdm(loader, desc="Val")
        for batch in pbar:
            input_ids = batch["input_ids"].to(dev)
            attn_mask = batch["attention_mask"].to(dev)
            labels = batch["labels"].to(dev)
            with autocast():
                out = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
                loss = out.loss
                logits = out.logits
            total_loss += loss.item()
            pred = torch.argmax(logits, dim=1)
            acc = accuracy_score(labels.cpu().numpy(), pred.cpu().numpy())
            total_acc += acc
            steps += 1
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
    return total_loss / steps, total_acc / steps

# ===================== 主训练循环 + 早停保存最优模型 =====================
logger.info("==== Start training ====")
best_val_acc = 0.0

for ep in range(1, epochs+1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, device)
    va_loss, va_acc = val_one_epoch(model, val_loader, device)
    cost = time.time() - t0
    msg = f"Epoch {ep:2d} | Train Loss:{tr_loss:.4f} Acc:{tr_acc:.4f} | Val Loss:{va_loss:.4f} Acc:{va_acc:.4f} | Time:{cost:.1f}s"
    logger.info(msg)
    print(msg)

    # 保存验证集最优权重
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        logger.info(f"Save best model, val_acc={best_val_acc:.4f}")

# 加载最优权重做预测
logger.info(f"Load best checkpoint val_acc={best_val_acc:.4f}")
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))

# ===================== 预测 =====================
logger.info("Start predicting test set ...")
model.eval()
preds_all = []
with torch.no_grad():
    pbar = tqdm(test_loader, desc="Predict")
    for batch in pbar:
        input_ids = batch["input_ids"].to(device)
        attn_mask = batch["attention_mask"].to(device)
        with autocast():
            out = model(input_ids=input_ids, attention_mask=attn_mask)
        logits = out.logits
        pred = torch.argmax(logits, dim=1).cpu()
        preds_all.extend(pred.numpy().tolist())

sample = pd.read_csv(SAMPLE_SUB_PATH)
sample["sentiment"] = preds_all
sample.to_csv(OUTPUT_CSV, index=False, quoting=3)
logger.info(f"Prediction finished, saved to {OUTPUT_CSV}, shape {sample.shape}")
print(f"Prediction finished, saved to {OUTPUT_CSV}, shape {sample.shape}")


2026-08-21 02:22:30,435 | INFO | === Logger initialized successfully ===
2026-08-21 02:22:30,436 | INFO | Running on device: cuda
Running on device: cuda
2026-08-21 02:22:30,437 | INFO | Loading training data ...
2026-08-21 02:22:31,046 | INFO | train shape = (25000, 3)
2026-08-21 02:22:31,047 | INFO | Loading test data ...
2026-08-21 02:22:31,672 | INFO | test shape = (25000, 2)
2026-08-21 02:22:31,930 | INFO | HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-08-21 02:22:31,998 | INFO | HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-08-21 02:22:32,065 | INFO | HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-08-21 02:22:32,130 | INFO | HTTP Request: GET https://huggi

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2026-08-21 02:22:55,881 | INFO | ==== Start training ====


/tmp/ipykernel_58/2008709253.py:114: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()  # 混合精度AMP
Train:   0%|          | 0/1250 [00:00<?, ?it/s]/tmp/ipykernel_58/2008709253.py:130: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Train:   0%|          | 1/1250 [00:00<09:16,  2.24it/s, loss=0.6882]/tmp/ipykernel_58/2008709253.py:130: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Train:   0%|          | 2/1250 [00:00<05:48,  3.58it/s, loss=0.7224]/tmp/ipykernel_58/2008709253.py:130: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Train:   0%|          | 3/1250 [00:00<04:36,  4.52it/s, loss=0.7014]/tmp/ipykerne

...

Train:   0%|          | 0/1250 [00:00<?, ?it/s]/tmp/ipykernel_58/2008709253.py:130: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Train:   0%|          | 1/1250 [00:00<02:49,  7.38it/s, loss=0.0845]/tmp/ipykernel_58/2008709253.py:130: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Train:   0%|          | 2/1250 [00:00<02:51,  7.27it/s, loss=0.1568]/tmp/ipykernel_58/2008709253.py:130: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Train:   0%|          | 3/1250 [00:00<02:52,  7.24it/s, loss=0.0782]/tmp/ipykernel_58/2008709253.py:130: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Train:   0%|          | 4/1250 [0

2026-08-21 02:32:10,377 | INFO | Epoch  3 | Train Loss:0.1350 Acc:0.9596 | Val Loss:0.4460 Acc:0.8858 | Time:183.8s


Epoch  3 | Train Loss:0.1350 Acc:0.9596 | Val Loss:0.4460 Acc:0.8858 | Time:183.8s
2026-08-21 02:32:11,232 | INFO | Save best model, val_acc=0.8858
2026-08-21 02:32:11,234 | INFO | Load best checkpoint val_acc=0.8858
2026-08-21 02:32:11,585 | INFO | Start predicting test set ...


Predict:   0%|          | 0/1563 [00:00<?, ?it/s]/tmp/ipykernel_58/2008709253.py:207: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Predict: 100%|██████████| 1563/1563 [00:49<00:00, 31.86it/s]

2026-08-21 02:33:00,698 | INFO | Prediction finished, saved to /kaggle/working/bert_submission.csv, shape (25000, 2)
Prediction finished, saved to /kaggle/working/bert_submission.csv, shape (25000, 2)


In [7]:
import os
import sys
import logging
import time
import random
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import GradScaler
from torch.amp import autocast
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from transformers import BertTokenizerFast, BertForSequenceClassification, get_linear_schedule_with_warmup
from tqdm import tqdm

# ===================== 全局随机种子，保证可复现 =====================
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ===================== 路径配置 =====================
TRAIN_ZIP_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip"
TEST_ZIP_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
SAMPLE_SUB_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/sampleSubmission.csv"
OUTPUT_CSV = "/kaggle/working/bert_submission.csv"
BEST_MODEL_PATH = "/kaggle/working/best_bert.pt"
os.makedirs("/kaggle/working", exist_ok=True)

# ==========日志修复 ==========
root_logger = logging.getLogger()
if root_logger.handlers:
    root_logger.handlers.clear()
logging.basicConfig(
    stream=sys.stdout,
    format='%(asctime)s | %(levelname)s | %(message)s',
    level=logging.INFO
)
program = os.path.basename(sys.argv[0])
logger = logging.getLogger(program)
logger.setLevel(logging.INFO)
logger.info("=== Logger initialized successfully ===")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Running on device: {device}")

# ===================== Dataset：接收已经encode好的数据 =====================
class ImdbDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# ===================== 加载数据 =====================
logger.info("Loading training data ...")
train_df = pd.read_csv(TRAIN_ZIP_PATH, header=0, delimiter="\t", quoting=3)
logger.info(f"train shape = {train_df.shape}")

logger.info("Loading test data ...")
test_df = pd.read_csv(TEST_ZIP_PATH, header=0, delimiter="\t", quoting=3)
logger.info(f"test shape = {test_df.shape}")

X_train, X_val, y_train, y_val = train_test_split(
    train_df["review"].tolist(),
    train_df["sentiment"].tolist(),
    test_size=0.2,
    random_state=SEED,
    stratify=train_df["sentiment"]
)

model_name = "bert-base-uncased"
tokenizer = BertTokenizerFast.from_pretrained(model_name)
max_seq_len = 256       # 提升序列长度，换取分数
batch_size = 8          # 256序列T4下调batch

logger.info("Pre‑tokenizing train/val/test texts ...")
train_enc = tokenizer(X_train, max_length=max_seq_len, truncation=True, padding="max_length")
val_enc = tokenizer(X_val, max_length=max_seq_len, truncation=True, padding="max_length")
test_enc = tokenizer(test_df["review"].tolist(), max_length=max_seq_len, truncation=True, padding="max_length")

train_dataset = ImdbDataset(train_enc, y_train)
val_dataset = ImdbDataset(val_enc, y_val)
test_dataset = ImdbDataset(test_enc, labels=None)

num_workers = 0
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

epochs = 3
lr = 1.5e-5
optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-5)

total_steps = len(train_loader) * epochs
warmup_steps = int(total_steps * 0.1)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

scaler = GradScaler()
grad_clip_norm = 1.0

# ===================== 训练验证函数：关闭tqdm逐batch输出刷屏 =====================
def train_one_epoch(model, loader, opt, sch, scaler, dev):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    steps = 0
    for batch in tqdm(loader, desc="Train", disable=False):
        input_ids = batch["input_ids"].to(dev)
        attn_mask = batch["attention_mask"].to(dev)
        labels = batch["labels"].to(dev)
        opt.zero_grad()

        with autocast('cuda'):
            out = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
            loss = out.loss
            logits = out.logits

        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)

        scaler.step(opt)
        scaler.update()
        sch.step()

        total_loss += loss.item()
        pred = torch.argmax(logits, dim=1)
        acc = accuracy_score(labels.cpu().numpy(), pred.cpu().numpy())
        total_acc += acc
        steps += 1
    return total_loss / steps, total_acc / steps

def val_one_epoch(model, loader, dev):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    steps = 0
    with torch.no_grad():
        for batch in tqdm(loader, desc="Val", disable=False):
            input_ids = batch["input_ids"].to(dev)
            attn_mask = batch["attention_mask"].to(dev)
            labels = batch["labels"].to(dev)
            with autocast('cuda'):
                out = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
                loss = out.loss
                logits = out.logits
            total_loss += loss.item()
            pred = torch.argmax(logits, dim=1)
            acc = accuracy_score(labels.cpu().numpy(), pred.cpu().numpy())
            total_acc += acc
            steps += 1
    return total_loss / steps, total_acc / steps

# ===================== 主训练循环，只打印Epoch汇总，不再刷屏逐batch日志 =====================
logger.info("==== Start training ====")
best_val_acc = 0.0

for ep in range(1, epochs+1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, device)
    va_loss, va_acc = val_one_epoch(model, val_loader, device)
    cost = time.time() - t0
    msg = f"Epoch {ep:2d} | Train Loss:{tr_loss:.4f} Acc:{tr_acc:.4f} | Val Loss:{va_loss:.4f} Acc:{va_acc:.4f} | Time:{cost:.1f}s"
    logger.info(msg)

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        logger.info(f"Save best model, val_acc={best_val_acc:.4f}")

logger.info(f"Load best checkpoint val_acc={best_val_acc:.4f}")
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))

# ===================== 预测 =====================
logger.info("Start predicting test set ...")
model.eval()
preds_all = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predict"):
        input_ids = batch["input_ids"].to(device)
        attn_mask = batch["attention_mask"].to(device)
        with autocast('cuda'):
            out = model(input_ids=input_ids, attention_mask=attn_mask)
        logits = out.logits
        pred = torch.argmax(logits, dim=1).cpu()
        preds_all.extend(pred.numpy().tolist())

sample = pd.read_csv(SAMPLE_SUB_PATH)
sample["sentiment"] = preds_all
sample.to_csv(OUTPUT_CSV, index=False, quoting=3)
logger.info(f"Prediction finished, saved to {OUTPUT_CSV}, shape {sample.shape}")


2026-08-21 02:38:33,811 | INFO | === Logger initialized successfully ===
2026-08-21 02:38:33,811 | INFO | Running on device: cuda
2026-08-21 02:38:33,812 | INFO | Loading training data ...
2026-08-21 02:38:34,395 | INFO | train shape = (25000, 3)
2026-08-21 02:38:34,396 | INFO | Loading test data ...
2026-08-21 02:38:34,965 | INFO | test shape = (25000, 2)
2026-08-21 02:38:35,113 | INFO | HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-08-21 02:38:35,174 | INFO | HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-08-21 02:38:35,239 | INFO | HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-08-21 02:38:35,305 | INFO | HTTP Request: GET https://huggingface.co/api/models/ber

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2026-08-21 02:38:56,200 | INFO | ==== Start training ====


/tmp/ipykernel_58/2685251633.py:111: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Val: 100%|██████████| 625/625 [00:20<00:00, 30.03it/s]

...

Val: 100%|██████████| 625/625 [00:20<00:00, 29.88it/s]

2026-08-21 02:57:50,135 | INFO | Epoch  3 | Train Loss:0.1141 Acc:0.9744 | Val Loss:0.4178 Acc:0.9172 | Time:377.1s


2026-08-21 02:57:51,003 | INFO | Save best model, val_acc=0.9172
2026-08-21 02:57:51,005 | INFO | Load best checkpoint val_acc=0.9172
2026-08-21 02:57:51,328 | INFO | Start predicting test set ...


Predict: 100%|██████████| 3125/3125 [01:43<00:00, 30.27it/s]

2026-08-21 02:59:34,625 | INFO | Prediction finished, saved to /kaggle/working/bert_submission.csv, shape (25000, 2)
